# Лабораторна робота №3
## Тема: Лінійна регресія у Scikit-learn: підготовка даних, навчання, прогнозування та оцінювання якості моделі
* **Мета:** Набути практичних навичок розв'язання задачі регресії; навчитися відокремлювати ознаки від цільової змінної, запобігати витоку даних, розділяти вибірку на навчальну і тестову частини, порівнювати модель із базовим прогнозом та інтерпретувати MAE, RMSE і коефіцієнт детермінації $R^2$.
* **Студент:** Заболотний Назар
* **Група:** МІТ-31 (Варіант / `random_state` = 8)

### Опис набору даних
* **Назва:** Student Performance (Mathematics)
* **Джерело:** UCI Machine Learning Repository / OpenML (`openml.org`, ID: 42352)
* **Ліцензія:** CC BY 4.0
* **Формулювання задачі:** За соціально-демографічними, сімейними та навчальними характеристиками учня спрогнозувати його підсумкову річну оцінку з математики (`G3`, шкала від 0 до 20 балів).

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

SEED = 8  # номер у списку групи

# Завантаження датасету Student Performance з OpenML
data = fetch_openml(data_id=42352, as_frame=True, parser="auto")
df = data.frame.copy()

print("Розмір датасету:", df.shape)
display(df.head())

/Users/nazarzabolotniy/Desktop/Data_Science_Labs/.venv/lib/python3.15/site-packages/sklearn/datasets/_openml.py:131: UserWarning: A network error occurred while downloading https://www.openml.org/api/v1/json/data/42352. Retrying...
  warn(


URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1147)>

In [ ]:
print("--- Типи даних (dtypes) ---")
print(df.dtypes)

print("\n--- Інформація про таблицю (info) ---")
df.info()

print("\n--- Описова статистика числових полів (describe) ---")
display(df.describe())

print("\nКількість пропусків:", df.isna().sum().sum())
print("Кількість повних дублікатів:", df.duplicated().sum())

# Словник ключових полів:
# G3 - фінальна оцінка (ціль, 0..20 балів)
# G1, G2 - оцінки за 1-й та 2-й періоди (вилучаємо, щоб уникнути витоку даних)
# studytime - тижневий час навчання (1..4)
# failures - кількість попередніх академічних заборгованостей
# absences - кількість пропусків занять
# age - вік учня (15..22 роки)
# school, sex, address, famsize, Pstatus, Mjob, Fjob, reason, guardian, higher, internet - категоріальні ознаки

df["G3"] = pd.to_numeric(df["G3"])

# Вилучаємо G1 та G2 (джерела витоку цільової змінної)
X = df.drop(columns=["G3", "G1", "G2"])
y = df["G3"]

print("\nФорма матриці ознак X:", X.shape)
print("Форма вектора цілі y:", y.shape)

### Пояснення до підготовки даних та контролю витоку
* Набір даних містить **395 записів** та **33 стовпці** без пропущених значень і дублікатів.
* Ознаки `G1` та `G2` (проміжні оцінки за перший і другий семестри) було **вилучено**, оскільки вони формуються безпосередньо перед підсумковою оцінкою `G3` і мають з нею майже лінійну залежність — їх залишення призвело б до **витоку даних (data leakage)** та нереалістичного завищення метрик при прогнозуванні на початку навчального року.

In [ ]:
# Поділ вибірки 80/20 ДО будь-яких перетворень
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED
)

print(f"Розмір train: {X_train.shape[0]} рядків, test: {X_test.shape[0]} рядків")

# Автоматичне визначення числових та категоріальних ознак
numeric_features = X_train.select_dtypes(include=["int64", "float64", "number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["int64", "float64", "number"]).columns.tolist()

print("Числові ознаки:", numeric_features)
print("Категоріальні ознаки:", categorical_features)

# Конвеєр підготовки даних
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessing = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

# 1. Базова модель (DummyRegressor)
baseline_model = Pipeline([
    ("preprocessing", preprocessing),
    ("regressor", DummyRegressor(strategy="mean"))
])

# 2. Основна модель (LinearRegression)
linreg_model = Pipeline([
    ("preprocessing", preprocessing),
    ("regressor", LinearRegression())
])

# Навчання лише на train
baseline_model.fit(X_train, y_train)
linreg_model.fit(X_train, y_train)

# Прогнозування на test
y_pred_dummy = baseline_model.predict(X_test)
y_pred_lr = linreg_model.predict(X_test)

def get_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    r2 = r2_score(y_true, y_pred)
    return {"MAE (бали)": mae, "RMSE (бали)": rmse, "R2": r2}

metrics_df = pd.DataFrame([
    get_metrics(y_test, y_pred_dummy),
    get_metrics(y_test, y_pred_lr)
], index=["DummyRegressor (mean)", "LinearRegression"])

print("\n--- Порівняння метрик на тестовій вибірці ---")
display(metrics_df.round(4))

In [ ]:
residuals = y_test - y_pred_lr

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# 1. Розподіл цільової змінної G3
sns.histplot(y, bins=15, kde=True, color="steelblue", ax=axes[0])
axes[0].set_title("Розподіл фінальної оцінки (G3)")
axes[0].set_xlabel("Оцінка G3, бали")
axes[0].set_ylabel("Кількість учнів")
axes[0].grid(True, alpha=0.3)

# 2. Фактичні значення проти прогнозованих
axes[1].scatter(y_test, y_pred_lr, alpha=0.75, color="teal", edgecolors="k", linewidth=0.5)
min_val, max_val = y_test.min(), y_test.max()
axes[1].plot([min_val, max_val], [min_val, max_val], "r--", lw=2, label="Ідеальний прогноз")
axes[1].set_title("Фактичні vs Прогнозовані (Test)")
axes[1].set_xlabel("Фактична оцінка G3, бали")
axes[1].set_ylabel("Прогнозована оцінка G3, бали")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# 3. Залишки відносно прогнозу
axes[2].scatter(y_pred_lr, residuals, alpha=0.75, color="darkorange", edgecolors="k", linewidth=0.5)
axes[2].axhline(0, color="black", linestyle="--", lw=1.5)
axes[2].set_title("Залишки відносно прогнозу")
axes[2].set_xlabel("Прогнозована оцінка G3, бали")
axes[2].set_ylabel("Залишок (Факт - Прогноз), бали")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("Lab3_plots.png", dpi=300)
plt.show()

# Збереження результатів
predictions_df = pd.DataFrame({
    "y_true": y_test.values,
    "y_pred_dummy": y_pred_dummy,
    "y_pred_lr": y_pred_lr,
    "residual": residuals.values
})
predictions_df.to_csv("test_predictions.csv", index=False)
metrics_df.to_csv("metrics.csv")
print("Файли test_predictions.csv, metrics.csv та Lab3_plots.png успішно збережено.")

## Висновок та інтерпретація результатів
1. **Порівняння з базовою моделлю:** Навчена модель `LinearRegression` перевершує базовий прогноз `DummyRegressor` (середнє значення), оскільки має нижчі показники помилок: **MAE = ... балів** проти **... балів** у baseline, **RMSE = ... балів** проти **... балів**, а коефіцієнт детермінації становить **$R^2 = ...$** (додатне значення свідчить, що модель пояснює частину дисперсії оцінок краще за просте середнє).
2. **Аналіз графіків і залишків:** На графіку розподілу цілі та залишків помітно групу учнів із нульовою підсумковою оцінкою (`G3 = 0`), для яких лінійна модель прогнозує значення в межах 6–10 балів. Саме на цих спостереженнях виникають найбільші від'ємні залишки (близько `-8...-10` балів), що суттєво збільшує `RMSE`.
3. **Обмеження та напрями покращення:** Соціально-демографічні фактори без поточних результатів навчання (`G1`, `G2`) лише частково визначають фінальний бал, а лінійна модель не враховує нелінійні взаємодії ознак. Покращити якість можна за рахунок регуляризації (Ridge/Lasso), відбору найважливіших ознак або використання дерев рішень та градієнтного бустингу.